In [ ]:
# ---------------- Imports ----------------
import os
import json

import pandas as pd
import yaml




In [ ]:
# ---------------- Args ----------------
FEVER_TRAIN_NAME = "train.jsonl"
FEVER_DEV_NAME = "shared_task_dev.jsonl"
FEVEROUS_TRAIN_NAME = "feverous_train_challenges.jsonl"
FEVEROUS_DEV_NAME = "feverous_dev_challenges.jsonl"



In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
DATA_FOLDER = os.path.join(PROJ_STORE, "data")

EXTERNAL_SETS_FOLDER = os.path.join(DATA_FOLDER, "external")

# FEVER
FEVER_PATH = os.path.join(EXTERNAL_SETS_FOLDER, "fever")
FEVER_TRAIN_PATH = os.path.join(FEVER_PATH, FEVER_TRAIN_NAME)
FEVER_DEV_PATH = os.path.join(FEVER_PATH, FEVER_DEV_NAME)

# FEVEROUS
FEVEROUS_PATH = os.path.join(EXTERNAL_SETS_FOLDER, "feverous")
FEVEROUS_TRAIN_PATH = os.path.join(FEVEROUS_PATH, FEVEROUS_TRAIN_NAME)
FEVEROUS_DEV_PATH = os.path.join(FEVEROUS_PATH, FEVEROUS_DEV_NAME)

# OUTPUT
OUTPUT_DIR = os.path.join(DATA_FOLDER, "combined-claims")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "combined-claims-full") # NOTE: file extension added on output time.



In [ ]:
# ---------------- Functions ----------------

# Checks
def check_duplicates(dataframe, column):
    dup_mask =  dataframe.duplicated(subset=[column], keep=False)
    return dataframe.loc[dup_mask]



In [ ]:
# ---------------- Workspace ----------------

# FEVER

cols = ["id", "claim", "label"]

## Train
fever_train_data = pd.read_json(FEVER_TRAIN_PATH, lines=True)[cols]

print(f"Fever train shape: {fever_train_data.shape}")

## Dev
fever_dev_data = pd.read_json(FEVER_DEV_PATH, lines=True)[cols]

print(f"Fever dev shape: {fever_dev_data.shape}")

## Full
fever_data = pd.concat([fever_train_data, fever_dev_data], ignore_index=True)

print(f"Raw full fever shape: {fever_data.shape}")

# Fix id
fever_data = fever_data.assign(id=lambda d: "fever-" + d["id"].astype(str))


# Drop NOT ENOUGH INFO rows
fever_data = fever_data[fever_data["label"] != "NOT ENOUGH INFO"].reset_index(drop=True)


# Drop duplicate claims
fever_data = fever_data.drop_duplicates(subset=["claim"]).reset_index(drop=True)

print(f"Processed full fever shape: {fever_data.shape}")

display(fever_data.head())


# Check
dup_rows = check_duplicates(fever_data, "id")
assert dup_rows.empty, f"Duplicated IDs found.\n{dup_rows.sort_values('id')}"
dup_claims = check_duplicates(fever_data, "claim")
assert dup_claims.empty, f"Duplicated claims found.\n{dup_claims.sort_values('claim')}"


In [ ]:
# FEVEROUS

cols = ["id", "claim", "label"]

## Train
feverous_train_data = pd.read_json(FEVEROUS_TRAIN_PATH, lines=True)[cols]

print(f"Feverous train shape: {feverous_train_data.shape}")

### Dev
feverous_dev_data = pd.read_json(FEVEROUS_DEV_PATH, lines=True)[cols]

print(f"Feverous dev shape: {feverous_dev_data.shape}")

## Full
feverous_data = pd.concat([feverous_train_data, feverous_dev_data], ignore_index=True)
print(f"Raw full fever shape: {feverous_data.shape}")


# Cleaning
feverous_data = feverous_data[feverous_data["id"].astype(str).str.len()>0] # drop where id is empty

# Fix id
feverous_data = feverous_data.assign(id=lambda d: "feverous-" + d["id"].astype(str))

# Drop NOT ENOUGH INFO rows
feverous_data = feverous_data[feverous_data["label"] != "NOT ENOUGH INFO"].reset_index(drop=True)


print(f"Processed full fever shape: {feverous_data.shape}")
display(feverous_data.head())



# Checks
dup_rows = check_duplicates(feverous_data, "id")
assert dup_rows.empty, f"Duplicated IDs found.\n{dup_rows.sort_values('id')}"
dup_claims = check_duplicates(feverous_data, "claim")
assert dup_claims.empty, f"Duplicated claims found.\n{dup_claims.sort_values('claim')}"



In [ ]:
# Combine all
claims_data = pd.concat([fever_data, feverous_data], ignore_index=True)


claims_data = claims_data.rename(columns={"id": "claim_id", "claim": "claim_text", "label": "true_label"}) 


print(f"Claims data shape: {claims_data.shape}")

display(claims_data.head())


# Checks
dup_rows = check_duplicates(claims_data, "claim_id")
assert dup_rows.empty, f"Duplicated IDs found.\n{dup_rows.sort_values('claim_id')}"
dup_claims = check_duplicates(claims_data, "claim_text")
assert dup_claims.empty, f"Duplicated claims found.\n{dup_claims.sort_values('claim_text')}"



In [ ]:
# Save full file AND split into 3 shards

NUM_SHARDS = 3

full_path = f"{OUTPUT_FILE}.jsonl"
shard_paths = [
    f"{OUTPUT_FILE}.shard{i}.jsonl" for i in range(NUM_SHARDS)
]

# Open files
with open(full_path, "w", encoding="utf-8") as full_f, \
     open(shard_paths[0], "w", encoding="utf-8") as shard0, \
     open(shard_paths[1], "w", encoding="utf-8") as shard1, \
     open(shard_paths[2], "w", encoding="utf-8") as shard2:

    shard_files = [shard0, shard1, shard2]

    for idx, record in enumerate(claims_data.to_dict(orient="records")):
        line = json.dumps(record, ensure_ascii=False) + "\n"

        # Write full dataset
        full_f.write(line)

        # Write shard
        shard_id = idx % NUM_SHARDS
        shard_files[shard_id].write(line)

print(f"Exported full JSONL to {full_path}")
print("Exported shard files:")
for p in shard_paths:
    print(f"- {p}")

